In [68]:
# จัดการข้อมูลและแสดงผล
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# แบ่งข้อมูลและสร้างโมเดล
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# จัดการค่าว่างและปรับสเกล
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ประเมินผลโมเดล
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score,
)

In [69]:
df = pd.read_csv('data_preprocessed_V1.csv')
df_test = pd.read_csv('test_preprocessed_V1.csv')

In [70]:
X = df[['Is_male', 'Is_married',
       'Declared Number of Borrowings',
       'Declared Borrowing Amount', 'Rent Burden',
       'Number of Dependents', 'Number of Dependent Children',
       'Home Phone Number Available', 'Mobile Phone Number Available',
       'Employer Phone Number Available', 'Driver Licence Number Available',
       'Log Declared Annual Income', 'Log Rent Burden',
       'Log Desired Credit Limit', 'Has_rent', 'Age_bins_25–34',
       'Age_bins_35–44', 'Age_bins_45–54', 'Age_bins_55–64', 'Age_bins_65+',
       'Application Channel_17', 'Application Channel_18',
       'Application Channel_20', 'Business Entity Type_1',
       'Business Entity Type_2', 'Business Entity Type_5',
       'Business Entity Type_6', 'Business Entity Type_7',
       'Business Entity Type_9', 'Company Size Category_2',
       'Company Size Category_3', 'Company Size Category_4',
       'Company Size Category_5', 'Company Size Category_6',
       'Company Size Category_7', 'Company Size Category_8',
       'Company Size Category_9', 'Employment Type Category_2',
       'Employment Type Category_3', 'Employment Type Category_4',
       'Employment Type Category_5', 'Employment Type Category_6',
       'Employment Type Category_7', 'Employment Type Category_8',
       'Industry Category_2', 'Industry Category_3', 'Industry Category_4',
       'Industry Category_5', 'Industry Category_6', 'Industry Category_7',
       'Industry Category_8', 'Industry Category_9', 'Industry Category_10',
       'Industry Category_11', 'Industry Category_12', 'Industry Category_13',
       'Industry Category_15', 'Industry Category_16', 'Industry Category_17',
       'Industry Category_18', 'Industry Category_99', 'Insurance Type_1',
       'Insurance Type_3', 'Insurance Type_4', 'Occupation Category_240',
       'Occupation Category_340', 'Occupation Category_440',
       'Occupation Category_540', 'Occupation Category_640',
       'Occupation Category_740', 'Occupation Category_840',
       'Occupation Category_940', 'Occupation Category_1040',
       'Occupation Category_1140', 'Occupation Category_1240',
       'Occupation Category_1340', 'Occupation Category_1440',
       'Occupation Category_1540', 'Payday_Period_Mid_month',
       'Payday_Period_Late_month', 'Residence Ownership Status_1',
       'Residence Ownership Status_2', 'Residence Ownership Status_3',
       'Residence Type_2', 'Residence Type_3', 'Residence Type_4',
       'Residence Type_5', 'Residence Type_6', 'Residence_age_bins_1–5 years',
       'Residence_age_bins_5–10 years', 'Residence_age_bins_10–20 years',
       'Residence_age_bins_20-40 years', 'Residence_age_bins_40-60 years',
       'Residence_age_bins_60+ years']]
y = df[['12-Month Default Flag']]

In [71]:
X_train,X_test,y_train,y_test = train_test_split(X,y,stratify=y,test_size=0.2,random_state=67)

In [72]:
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ให้ y เป็น 1 มิติ รองรับทั้ง Series และ DataFrame คอลัมน์เดียว
y_train_1d = np.asarray(y_train).ravel()
y_test_1d = np.asarray(y_test).ravel()

print(f"สัดส่วนคลาส 1 ใน train: {(y_train_1d == 1).mean():.1%}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(solver="lbfgs", max_iter=1000)),
])

search = GridSearchCV(
    estimator=pipeline,
    param_grid={
        "model__C": [0.001, 0.01, 0.1, 1, 10],
        "model__class_weight": [
            None,
            {0: 1, 1: 2},
            {0: 1, 1: 4},
            "balanced",
        ],
    },
    scoring={"roc_auc": "roc_auc", "ap": "average_precision"},
    refit="roc_auc",
    cv=cv,
)
search.fit(X_train, y_train_1d)

best_model = search.best_estimator_
default_idx = list(best_model.classes_).index(1)

# แต่ละแถวได้คะแนนจากโมเดลที่ไม่ได้ฝึกด้วยแถวนั้น
oof_prob = cross_val_predict(
    best_model, X_train, y_train_1d, cv=cv, method="predict_proba"
)[:, default_idx]

precision, recall, thresholds = precision_recall_curve(y_train_1d, oof_prob)
p, r = precision[:-1], recall[:-1]

# F2 ให้น้ำหนัก recall มากกว่า precision
denominator = 4 * p + r
f2 = np.divide(
    5 * p * r,
    denominator,
    out=np.zeros_like(p),
    where=denominator > 0,
)
threshold = thresholds[np.argmax(f2)]

# ประเมินบน holdout หลังเลือกโมเดลและ threshold แล้ว
y_prob = best_model.predict_proba(X_test)[:, default_idx]
y_pred = (y_prob >= threshold).astype(int)

print("Best parameters:", search.best_params_)
print(f"Mean CV ROC-AUC:       {search.best_score_:.4f}")
print(f"Selected threshold:    {threshold:.4f}")
print(f"Test ROC-AUC:          {roc_auc_score(y_test_1d, y_prob):.4f}")
print(f"Test Average Precision: {average_precision_score(y_test_1d, y_prob):.4f}")
print(confusion_matrix(y_test_1d, y_pred, labels=[0, 1]))
print(classification_report(y_test_1d, y_pred, zero_division=0))

สัดส่วนคลาส 1 ใน train: 9.7%
Best parameters: {'model__C': 0.1, 'model__class_weight': {0: 1, 1: 2}}
Mean CV ROC-AUC:       0.6340
Selected threshold:    0.1202
Test ROC-AUC:          0.6369
Test Average Precision: 0.1517
[[3296 9087]
 [ 151 1176]]
              precision    recall  f1-score   support

           0       0.96      0.27      0.42     12383
           1       0.11      0.89      0.20      1327

    accuracy                           0.33     13710
   macro avg       0.54      0.58      0.31     13710
weighted avg       0.87      0.33      0.40     13710



In [73]:
logistic = best_model.named_steps["model"]
feature_names = best_model[:-1].get_feature_names_out()

coef_table = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": logistic.coef_[0],
})

coef_table["Abs Coefficient"] = coef_table["Coefficient"].abs()
coef_table = coef_table.sort_values("Abs Coefficient", ascending=False)

display(coef_table.head(20))
print(f"Intercept: {logistic.intercept_[0]:.4f}")

,Feature,Coefficient,Abs Coefficient
2,Declared Number of Borrowings,0.242941,0.242941
34,Company Size Category_7,0.228858,0.228858
22,Application Channel_20,-0.209844,0.209844
33,Company Size Category_6,0.176848,0.176848
16,Age_bins_35–44,-0.171403,0.171403
31,Company Size Category_4,0.158120,0.158120
32,Company Size Category_5,0.154405,0.154405
13,Log Desired Credit Limit,0.152458,0.152458
14,Has_rent,0.148514,0.148514
35,Company Size Category_8,0.143284,0.143284


Intercept: -1.6493


In [74]:
X_submit = df_test.reindex(columns=X_train.columns, fill_value=0)
default_idx = list(best_model.classes_).index(1)
probabilities = best_model.predict_proba(X_submit)[:, default_idx]

In [75]:
from pathlib import Path

ids = (
    df_test["ID"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0+$", "", regex=True)
    .reset_index(drop=True)
)

predictions = (np.asarray(probabilities) >= threshold).astype(int)

submission = pd.DataFrame({
    "ID": ids,
    "12-Month Default Flag": predictions,
})

output_path = Path("./submission/submission_V1.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output_path, index=False, encoding="utf-8")

print(f"Saved {len(submission):,} rows to {output_path}")

Saved 137,091 rows to submission\submission_V1.csv
